In [3]:
import os
import glob
import numpy as np
import pandas as pd
import dask.dataframe as dd #pandas mejorado
import matplotlib.pyplot as plt
import seaborn as sns
import math
from matplotlib import rc, rcParams
import matplotlib.gridspec as gridspec
from scipy.spatial import distance
from scipy.interpolate import make_interp_spline
import joblib
from mpl_toolkits.mplot3d import Axes3D

#astropy
from astropy.io import ascii, fits
from astropy.table import Table , vstack, hstack
from astropy import units as u
from astropy import cosmology
from astropy.cosmology import LambdaCDM, Planck18, Planck15
from astropy.coordinates import SkyCoord, Distance
from astropy.coordinates import ICRS, Galactic, FK4, FK5 
from astropy.coordinates import Angle, Latitude, Longitude  
from astropy.stats import sigma_clip
from astropy.constants import c

#sklearn-ML
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline
from sklearn.cluster import DBSCAN
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors, KernelDensity
import xgboost as xgb
from sklearn.tree import DecisionTreeRegressor 
from sklearn.ensemble import RandomForestClassifier, AdaBoostRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split, cross_validate, KFold, StratifiedKFold, learning_curve,cross_val_predict, GridSearchCV, cross_val_predict, cross_val_score, LeaveOneGroupOut
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import precision_recall_curve, classification_report, accuracy_score, confusion_matrix, roc_auc_score, roc_curve, recall_score, f1_score, precision_score, average_precision_score
from sklearn.utils.class_weight import compute_class_weight

In [4]:
path_file_name = 'D:\\Santa María\\10mo semestre\\Tesis\\Mocks\\'
path_cl_cat = 'D:\\Santa María\\10mo semestre\\Tesis\\'
cl_cat = np.loadtxt(path_cl_cat + 'clusters_catalogues_test.dat')  # M200, R200, RA, DEC, redshift_S

# LCDM
OMegaM = 0.3089
OmegaL = 0.6911
h = 0.6774
cosmo = LambdaCDM(H0=h * 100, Om0=OMegaM, Ode0=OmegaL)

# ===============================================================
# READ MOCKS
# ===============================================================

file_names = sorted(glob.glob(path_file_name + 'CHANCES_lowz_*_with_true_subs.fits'))

Clust_name, X_cent, Y_cent, Z_cent, RA, DEC, M_VIR, R_200_MPC, R_200_DEG, REDSHIFT = [], [], [], [], [], [], [], [], [], []
REDSHIFT_TRUE_MEAN, SIGMA_R200, MEMBERS_R200, HALO_MAX_MEMB, ALL_MEMBERS = [], [], [], [], []

for file in file_names:
    cl_id = file.split("_")[-4]  #  Extracts cluster index
    count = int(cl_id) if len(cl_id) == 2 else int(cl_id[-1])
    
    mock = Table.read(file, format='fits')  #r ead

    # Cluster properties
    M_vir = 10 ** cl_cat[count][0]
    R_200_mpc = cl_cat[count][1] / 1e3
    redshift = cl_cat[count][4]
    ra_cent_cl, dec_cent_cl = cl_cat[count][2], cl_cat[count][3]

    # center
    mock_center = mock[(mock['RA'] == ra_cent_cl) & (mock['DEC'] == dec_cent_cl)]
    x_cent_cl, y_cent_cl, z_cent_cl = mock_center['x'][0], mock_center['y'][0], mock_center['z'][0]

    # Relative coords
    mock['x0'] = mock['x'] - x_cent_cl
    mock['y0'] = mock['y'] - y_cent_cl
    mock['z0'] = mock['z'] - z_cent_cl
    r_pos = np.sqrt(mock['x0'] ** 2 + mock['y0'] ** 2 + mock['z0'] ** 2)
    mock['Dist_Mpc'] = r_pos

    # Projected disttance
    cg = SkyCoord(mock['RA'], mock['DEC'], unit='deg', frame='icrs')
    cc = SkyCoord(ra_cent_cl, dec_cent_cl, unit='deg', frame='icrs')
    sep_deg = cg.separation(cc).deg
    sep_mpc = ((cosmo.kpc_proper_per_arcmin(redshift) * sep_deg * 60 * u.arcmin).value) / 1e3
    mock['D_projected_deg'] = sep_deg
    mock['D_projected_Mpc'] = sep_mpc

    # Members within 5R200
    dist_5r200 = 5 * R_200_mpc
    membership = (mock['Dist_Mpc'] <= dist_5r200).astype(int)
    mock['True_Members'] = membership

    mock_cl = mock[mock['Dist_Mpc'] <= dist_5r200]
    mock_cl_r200 = mock[mock['Dist_Mpc'] <= R_200_mpc]

    # Compute velocity dispersion 
    c_km_s = (c / (1e3 * u.m) * u.km).value
    redshift_cl = redshift
    sigma_r200 = np.std(mock_cl_r200['redshift_S_1'] * c_km_s)
    members_r200 = len(mock_cl_r200)

    # Save cluster parameters
    Clust_name.append(cl_id)
    X_cent.append(x_cent_cl)
    Y_cent.append(y_cent_cl)
    Z_cent.append(z_cent_cl)
    RA.append(ra_cent_cl)
    DEC.append(dec_cent_cl)
    M_VIR.append(M_vir / 1e14)
    R_200_MPC.append(R_200_mpc)
    R_200_DEG.append((cosmo.arcsec_per_kpc_proper(redshift) * R_200_mpc * 1e3 * u.kpc / (3600 * u.arcsec)).value)
    REDSHIFT.append(redshift)
    REDSHIFT_TRUE_MEAN.append(np.mean(mock_cl['redshift_S_1']))
    SIGMA_R200.append(sigma_r200)
    MEMBERS_R200.append(members_r200)
    HALO_MAX_MEMB.append(np.max(mock_cl['upid']))
    ALL_MEMBERS.append(len(mock_cl))

    # Save true members CSV
    csv_output = file.replace('.fits', '_true_members.csv')
    mock_cl.write(csv_output, format='csv', overwrite=True)
    print(f'{csv_output} done')

# ===============================================================
# CREATE FINAL CLUSTER CATALOG
# ===============================================================

catalog_data = {
    'Cluster': Clust_name,
    'X_cent': X_cent,
    'Y_cent': Y_cent,
    'Z_cent': Z_cent,
    'RA': RA,
    'Dec': DEC,
    'M_200_1e14Mo': M_VIR,
    'R_200_mpc': R_200_MPC,
    'R_200_deg': R_200_DEG,
    'redshift': REDSHIFT,
    'redshift_true_mean': REDSHIFT_TRUE_MEAN,
    'sigma_true_r200': SIGMA_R200,
    'memb_true_r200': MEMBERS_R200,
    'halo_max_memb': HALO_MAX_MEMB,
    'all_members': ALL_MEMBERS,
}

df_catalog = pd.DataFrame(catalog_data)
df_catalog.to_csv('Mock-clusters_True_information.csv', index=False)
print('\n Done')

D:\Santa María\10mo semestre\Tesis\Mocks\CHANCES_lowz_00_with_true_subs_true_members.csv listo


D:\Santa María\10mo semestre\Tesis\Mocks\CHANCES_lowz_01_with_true_subs_true_members.csv listo


D:\Santa María\10mo semestre\Tesis\Mocks\CHANCES_lowz_02_with_true_subs_true_members.csv listo


D:\Santa María\10mo semestre\Tesis\Mocks\CHANCES_lowz_03_with_true_subs_true_members.csv listo


D:\Santa María\10mo semestre\Tesis\Mocks\CHANCES_lowz_04_with_true_subs_true_members.csv listo


...

D:\Santa María\10mo semestre\Tesis\Mocks\CHANCES_lowz_45_with_true_subs_true_members.csv listo


D:\Santa María\10mo semestre\Tesis\Mocks\CHANCES_lowz_47_with_true_subs_true_members.csv listo


D:\Santa María\10mo semestre\Tesis\Mocks\CHANCES_lowz_50_with_true_subs_true_members.csv listo


D:\Santa María\10mo semestre\Tesis\Mocks\CHANCES_lowz_51_with_true_subs_true_members.csv listo


D:\Santa María\10mo semestre\Tesis\Mocks\CHANCES_lowz_52_with_true_subs_true_members.csv listo

 Listo


In [5]:
df_catalog

,Cluster,X_cent,Y_cent,Z_cent,RA,Dec,M_200_1e14Mo,R_200_mpc,R_200_deg,redshift,redshift_true_mean,sigma_true_r200,memb_true_r200,halo_max_memb,all_members
0,00,-214.186523,42.010036,111.320641,348.903015,-27.022461,8.318565,1.709000,0.409673,0.057921,0.058463,951.475459,261,247390211488899,1269
1,01,169.337173,-6.052490,21.813995,177.952988,-7.335777,8.294943,1.707460,0.594353,0.038998,0.039460,732.091972,266,26800690013414,1290
2,02,47.178329,-166.445312,150.634781,105.825195,-41.046387,4.021260,1.341220,0.358481,0.051556,0.052137,700.897148,161,12403952360609,735
3,03,157.623276,-22.852051,-35.163818,171.750793,12.450005,1.257011,0.910310,0.315255,0.039208,0.039355,456.408819,34,27453522310824,152
4,04,-1.225098,-218.703857,-59.890625,89.679054,15.314400,2.415116,1.131560,0.303829,0.051306,0.050985,587.867323,74,274156441700662,379
5,05,-110.038330,-101.343506,142.306900,42.644562,-43.569584,2.359021,1.122750,0.320530,0.048068,0.048004,507.845801,62,274190799554244,271
6,07,-122.453369,24.554623,44.339832,348.661316,-19.546310,5.643635,1.488970,0.654778,0.030557,0.031533,937.395490,244,261134095357379,1680
7,09,98.766762,61.698402,81.695900,211.992523,-35.050758,2.894892,1.191810,0.475150,0.033839,0.032958,579.162443,134,81969250,361
8,10,-111.647461,220.586075,33.200176,296.845795,-7.648376,16.238546,2.135650,0.517579,0.057245,0.057634,1154.455911,690,261821296469654,2421
9,11,84.682022,238.779144,-79.155518,250.473160,17.350609,6.802476,1.598220,0.368694,0.060361,0.060106,744.687489,268,1340123748663,1278


In [6]:
df_catalog.describe()

,X_cent,Y_cent,Z_cent,RA,Dec,M_200_1e14Mo,R_200_mpc,R_200_deg,redshift,redshift_true_mean,sigma_true_r200,memb_true_r200,halo_max_memb,all_members
count,42.000000,42.000000,42.000000,42.000000,42.000000,42.000000,42.000000,42.000000,42.000000,42.000000,42.000000,42.000000,4.200000e+01,42.000000
mean,4.741504,-27.706654,26.753336,162.169969,-8.023296,3.794244,1.254813,0.484657,0.042659,0.042737,662.701775,140.071429,1.215575e+14,556.476190
std,122.024666,101.585625,113.772606,100.888051,38.258319,2.821685,0.271213,0.265027,0.015648,0.015605,166.229280,116.919483,1.259045e+14,473.852722
min,-214.186523,-235.710205,-188.101562,9.598877,-77.995262,1.193090,0.886961,0.214142,0.010903,0.011419,392.543361,25.000000,8.143004e+07,152.000000
25%,-111.245178,-55.709290,-55.557922,90.361284,-38.332609,2.003247,1.059557,0.311324,0.034404,0.033814,536.924379,67.000000,1.309115e+13,249.000000
50%,14.211714,-25.597900,27.507086,153.379822,-7.686310,3.010776,1.212720,0.368076,0.046260,0.046370,643.383133,101.500000,2.714429e+13,378.000000
75%,95.245577,24.425747,112.034323,225.179871,19.903925,4.625410,1.401485,0.533952,0.056307,0.056295,741.538609,166.250000,2.611341e+14,692.750000
max,213.538528,238.779144,252.283737,353.147949,73.782883,16.238546,2.135650,1.259349,0.064689,0.064927,1154.455911,690.000000,2.748436e+14,2421.000000
